In [1]:
import pandas as pd, numpy as np
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
from lsff_utils import paths, gbd_data
import pathlib

In [2]:
location = "India"
vehicle = "rice"

In [3]:
# Parameters
location = "nigeria"
vehicle = "wheat"


In [4]:
location = location.title()

In [5]:
# Simulation output lives in per-run, timestamp-named directories, so the run
# is resolved rather than hardcoded -- this is the same run the Snakefile's
# marker names, so the notebook and the workflow cannot disagree.
maternal_results = paths.latest_results(paths.MATERNAL_RESULTS_ROOT, location, vehicle)
maternal_results

PosixPath('/mnt/share/code/ndbs/vivarium_gates_lsff_2026/0200_pregnancy_sim/sim_results/wheat/nigeria/2026_10_09_23_05_42/results')

In [6]:
with gbd_data.quiet_gbd_logs():
    pop = vivarium_inputs.get_population_structure(location).value
pop[pop > 0]

location  sex     age_start  age_end     year_start  year_end
Nigeria   Female  0.000000   0.019178    2023        2024        7.907591e+04
                  0.019178   0.076712    2023        2024        2.337381e+05
                  0.076712   0.500000    2023        2024        1.706685e+06
                  0.500000   1.000000    2023        2024        1.972513e+06
                  1.000000   2.000000    2023        2024        3.899171e+06
                  2.000000   5.000000    2023        2024        1.129739e+07
                  5.000000   10.000000   2023        2024        1.766635e+07
                  10.000000  15.000000   2023        2024        1.651034e+07
                  15.000000  20.000000   2023        2024        1.483548e+07
                  20.000000  25.000000   2023        2024        1.222554e+07
                  25.000000  30.000000   2023        2024        9.984802e+06
                  30.000000  35.000000   2023        2024        7.985380e+06
  

In [7]:
with gbd_data.quiet_gbd_logs():
    asfr = vivarium_inputs.get_measure(
        gbd_mapping.covariates.age_specific_fertility_rate, "estimate", location
    ).value
asfr[asfr > 0]

location  sex     age_start  age_end  year_start  year_end  parameter  
Nigeria   Female  10.0       15.0     2023        2024      lower_value    0.002556
                                                            mean_value     0.005714
                                                            upper_value    0.011863
                  15.0       20.0     2023        2024      lower_value    0.060154
                                                            mean_value     0.070244
                                                            upper_value    0.082133
                  20.0       25.0     2023        2024      lower_value    0.171875
                                                            mean_value     0.194484
                                                            upper_value    0.219213
                  25.0       30.0     2023        2024      lower_value    0.195823
                                                            mean_value     0.208935
    

In [8]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
asfr

location  sex     age_start  age_end     year_start  year_end
Nigeria   Female  0.000000   0.019178    2023        2024        0.000000
                  0.019178   0.076712    2023        2024        0.000000
                  0.076712   0.500000    2023        2024        0.000000
                  0.500000   1.000000    2023        2024        0.000000
                  1.000000   2.000000    2023        2024        0.000000
                  2.000000   5.000000    2023        2024        0.000000
                  5.000000   10.000000   2023        2024        0.000000
                  10.000000  15.000000   2023        2024        0.005714
                  15.000000  20.000000   2023        2024        0.070244
                  20.000000  25.000000   2023        2024        0.194484
                  25.000000  30.000000   2023        2024        0.208935
                  30.000000  35.000000   2023        2024        0.194076
                  35.000000  40.000000   2023     

In [9]:
with gbd_data.quiet_gbd_logs():
    sbr = vivarium_inputs.get_measure(
        gbd_mapping.covariates.stillbirth_28_weeks_to_live_birth_ratio, "estimate", location
    ).value
sbr[sbr > 0]

location  year_start  year_end  parameter  
Nigeria   2023        2024      lower_value    0.020725
                                mean_value     0.026341
                                upper_value    0.032816
Name: value, dtype: float64

In [10]:
sbr = sbr.iloc[0]
sbr

0.0207252032270903

In [11]:
with gbd_data.quiet_gbd_logs():
    maternal_abortion_miscarriage_incidence = vivarium_inputs.get_measure(
        gbd_mapping.causes.maternal_abortion_and_miscarriage,
        "incidence_rate",
        location,
    ).mean(axis=1)
maternal_abortion_miscarriage_incidence[maternal_abortion_miscarriage_incidence > 0]

/ihme/homes/ndbs/miniconda3/envs/vivarium_gates_lsff_2026_artifact/lib/python3.11/site-packages/vivarium_inputs/utilities.py:508: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[f"{split_column_prefix}_end"] = [
quiet_gbd_logs: 6 data-quality warning(s) from vivarium_inputs, 3 distinct:
      2x  Data returned extra columns: {'version_id'}.
      2x  Data contains additional age groups beyond those specified by restriction range.
      2x  Data is missing the following expected sex ids: {1}.
  Set gbd_data.DEFAULT_VERBOSITY = gbd_data.Verbosity.WARNINGS to see these live, or Verbosity.QUIET to stop reporting them.


location  sex     age_start  age_end  year_start  year_end
Nigeria   Female  10.0       15.0     2023        2024        0.006976
                  15.0       20.0     2023        2024        0.074024
                  20.0       25.0     2023        2024        0.177579
                  25.0       30.0     2023        2024        0.164486
                  30.0       35.0     2023        2024        0.146138
                  35.0       40.0     2023        2024        0.103056
                  40.0       45.0     2023        2024        0.063297
                  45.0       50.0     2023        2024        0.035266
                  50.0       55.0     2023        2024        0.003660
dtype: float64

In [12]:
with gbd_data.quiet_gbd_logs():
    ectopic_pregnancy_incidence = vivarium_inputs.get_measure(
        gbd_mapping.causes.ectopic_pregnancy,
        "incidence_rate",
        location,
    ).mean(axis=1)
ectopic_pregnancy_incidence[ectopic_pregnancy_incidence > 0]

/ihme/homes/ndbs/miniconda3/envs/vivarium_gates_lsff_2026_artifact/lib/python3.11/site-packages/vivarium_inputs/utilities.py:508: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[f"{split_column_prefix}_end"] = [
quiet_gbd_logs: 6 data-quality warning(s) from vivarium_inputs, 3 distinct:
      2x  Data returned extra columns: {'version_id'}.
      2x  Data contains additional age groups beyond those specified by restriction range.
      2x  Data is missing the following expected sex ids: {1}.
  Set gbd_data.DEFAULT_VERBOSITY = gbd_data.Verbosity.WARNINGS to see these live, or Verbosity.QUIET to stop reporting them.


location  sex     age_start  age_end  year_start  year_end
Nigeria   Female  10.0       15.0     2023        2024        0.000623
                  15.0       20.0     2023        2024        0.002269
                  20.0       25.0     2023        2024        0.003786
                  25.0       30.0     2023        2024        0.003659
                  30.0       35.0     2023        2024        0.003887
                  35.0       40.0     2023        2024        0.003440
                  40.0       45.0     2023        2024        0.003382
                  45.0       50.0     2023        2024        0.002765
                  50.0       55.0     2023        2024        0.000730
dtype: float64

In [13]:
pregnancy_incidence = (
    asfr
    + (asfr * sbr)
    + maternal_abortion_miscarriage_incidence
    + ectopic_pregnancy_incidence
)
pregnancy_incidence[pregnancy_incidence > 0]

location  sex     age_start  age_end  year_start  year_end
Nigeria   Female  10.0       15.0     2023        2024        0.013432
                  15.0       20.0     2023        2024        0.147993
                  20.0       25.0     2023        2024        0.379880
                  25.0       30.0     2023        2024        0.381409
                  30.0       35.0     2023        2024        0.348123
                  35.0       40.0     2023        2024        0.236532
                  40.0       45.0     2023        2024        0.135560
                  45.0       50.0     2023        2024        0.069797
                  50.0       55.0     2023        2024        0.007271
dtype: float64

In [14]:
pregnancies = pop * pregnancy_incidence
pregnancies[pregnancies > 0]

location  sex     age_start  age_end  year_start  year_end
Nigeria   Female  10.0       15.0     2023        2024        2.217623e+05
                  15.0       20.0     2023        2024        2.195549e+06
                  20.0       25.0     2023        2024        4.644232e+06
                  25.0       30.0     2023        2024        3.808295e+06
                  30.0       35.0     2023        2024        2.779893e+06
                  35.0       40.0     2023        2024        1.547675e+06
                  40.0       45.0     2023        2024        7.258899e+05
                  45.0       50.0     2023        2024        3.159267e+05
                  50.0       55.0     2023        2024        2.820465e+04
dtype: float64

In [15]:
total_pregnancies = pregnancies.sum()
total_pregnancies

16267427.202231383

In [16]:
sim_population = (
    pd.read_parquet(
        paths.measure_path(maternal_results, "pregnancy_outcome_count")
    )
    .groupby(["input_draw", "scenario"])
    .value.sum()
    .mean()
)
sim_population

10000000.0

In [17]:
scalar = total_pregnancies / sim_population
scalar

1.6267427202231384

In [18]:
for result in [
    "ylds",
    "ylls",
    "pregnancy_outcome_count",
    "person_time_anemia",
    "transition_count_maternal_disorders",
]:
    df = pd.read_parquet(
        paths.measure_path(maternal_results, result)
    )
    df.value *= scalar
    path = pathlib.Path(
        f"../results/rescaled_pregnancy_results/{vehicle}/{location.lower()}/{result}.parquet"
    )
    path.parent.mkdir(exist_ok=True, parents=True)
    df.to_parquet(path)